## Transformers END-To-END
## English ---> Telugu

In [58]:
data = [
    ("i am a student", "నేను ఒక విద్యార్థిని"),
    ("how are you", "మీరు ఎలా ఉన్నారు"),
    ("i love machine learning", "నాకు మెషిన్ లెర్నింగ్ అంటే ఇష్టం"),
    ("good morning", "శుభోదయం"),
    ("thank you", "ధన్యవాదాలు"),
    ("see you later", "తర్వాత కలుద్దాం"),
    ("what is your name", "మీ పేరు ఏమిటి"),
    ("where are you going", "మీరు ఎక్కడికి వెళ్తున్నారు"),
    ("i like coffee", "నాకు కాఫీ ఇష్టం"),
    ("welcome", "స్వాగతం")
]

In [59]:
# Import necessary libraries

import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)
from tensorflow.keras import Model

In [60]:
# Separate Input and Output Sentences
english_sentences = [x[0] for x in data]

# Add START and END tokens
telugu_sentences = [
    "start " + x[1] + " end"
    for x in data
]

In [61]:
# Tokenization Parameters
vocab_size = 1000
sequence_length = 20

# English Tokenizer
source_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Telugu Tokenizer
target_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

In [62]:
# Learn Vocabulary
source_vectorization.adapt(english_sentences)
target_vectorization.adapt(telugu_sentences)

# Convert Text to Numbers
encoder_inputs = source_vectorization(english_sentences)
target_tokens = target_vectorization(telugu_sentences)

# Decoder Inputs and Targets
decoder_inputs = target_tokens[:, :-1]
decoder_targets = target_tokens[:, 1:]

In [63]:
# Positional Embedding Layer

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = Embedding(
            vocab_size,
            embed_dim
        )

        self.position_embedding = Embedding(
            sequence_length,
            embed_dim
        )
        self.sequence_length = sequence_length

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        embedded_tokens = self.token_embedding(inputs)
        embedded_positions = self.position_embedding(positions)
        return embedded_tokens + embedded_positions

In [64]:
# Encoder Block

class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(
        self,
        embed_dim,
        dense_dim,
        num_heads
    ):
        super().__init__()
        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.dense_proj = tf.keras.Sequential([
            Dense(
                dense_dim,
                activation="relu"
            ),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):
        attention_output = self.attention(
            inputs,
            inputs
        )

        proj_input = self.layernorm1(
            inputs + attention_output
        )

        proj_output = self.dense_proj(
            proj_input
        )

        return self.layernorm2(
            proj_input + proj_output
        )

In [65]:
# Decoder Block

class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(
        self,
        embed_dim,
        dense_dim,
        num_heads
    ):
        super().__init__()
        self.self_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.cross_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(
                dense_dim,
                activation="relu"
            ),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()
        self.layernorm3 = LayerNormalization()
    def call(
        self,
        inputs,
        encoder_outputs
    ):

        attention_output = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            use_causal_mask=True
        )

        out1 = self.layernorm1(
            inputs + attention_output
        )
# Decoder Block

class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(
        self,
        embed_dim,
        dense_dim,
        num_heads
    ):
        super().__init__()
        self.self_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.cross_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(
                dense_dim,
                activation="relu"
            ),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()
        self.layernorm3 = LayerNormalization()
    def call(
        self,
        inputs,
        encoder_outputs
    ):

        attention_output = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            use_causal_mask=True
        )

        out1 = self.layernorm1(
            inputs + attention_output
        )

        attention_output2 = self.cross_attention(
            query=out1,
            value=encoder_outputs,
            key=encoder_outputs
        )

        out2 = self.layernorm2(
            out1 + attention_output2
        )

        ffn_output = self.ffn(out2)
        return self.layernorm3(
            out2 + ffn_output
        )
        attention_output2 = self.cross_attention(
            query=out1,
            value=encoder_outputs,
            key=encoder_outputs
        )

        out2 = self.layernorm2(
            out1 + attention_output2
        )

        ffn_output = self.ffn(out2)
        return self.layernorm3(
            out2 + ffn_output
        )

In [66]:
# Model Parameters

embed_dim = 128
dense_dim = 256
num_heads = 4

In [67]:
# Encoder Input

encoder_input = tf.keras.Input(
    shape=(None,),
    dtype="int64"
)

x = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(encoder_input)

encoder_output = TransformerEncoder(
    embed_dim,
    dense_dim,
    num_heads
)(x)

In [68]:
# Decoder Input

decoder_input = tf.keras.Input(
    shape=(None,),
    dtype="int64"
)

x = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(decoder_input)

x = TransformerDecoder(
    embed_dim,
    dense_dim,
    num_heads
)(x,encoder_output)

decoder_output = Dense(
    vocab_size,
    activation="softmax"
)(x)

In [82]:
# Create Transformer Model

transformer = Model(
    [encoder_input, decoder_input],
    decoder_output
)

# Compile Model

transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train Model

transformer.fit(
    [encoder_inputs, decoder_inputs],
    decoder_targets,
    batch_size=2,
    epochs=50
)

Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9947 - loss: 0.2335
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9842 - loss: 0.2196 
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9842 - loss: 0.1998 
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1798 
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1659 
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1538 
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1422 
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1318 
Epoch 9/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1225 
Epoch 10/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1139 
Epoch 11/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.1057 
Epoch 12/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0986 
Ep

In [85]:
# Prediction

test_sentence = ["i like coffee"]
encoder_input_test = source_vectorization(
    test_sentence
)

decoded_sentence = "start"
index_lookup = dict(
zip(range(len(target_vectorization.get_vocabulary())),target_vectorization.get_vocabulary()))

for i in range(sequence_length - 1):
    tokenized_target = target_vectorization(
        [decoded_sentence]

    )[:, :-1]

    predictions = transformer.predict(
        [encoder_input_test, tokenized_target],verbose=0

    )

    current_pos = len(decoded_sentence.split()) - 1

    sampled_token_index = np.argmax(
        predictions[0, current_pos, :]
    )

    sampled_token = index_lookup[
        sampled_token_index
    ]

    decoded_sentence += " " + sampled_token

    if sampled_token == "end":
        break

print("\nEnglish :", test_sentence[0])
print("Telugu  :", decoded_sentence)


English : i like coffee
Telugu  : start నాకు కాఫీ ఇష్టం end
